In [1]:
import numpy as np 
import pandas as pd 

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


from sklearn.model_selection import StratifiedKFold, cross_val_score


In [3]:
df = pd.read_csv(r'../datasets/breast-cancer.csv')

print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nDiagnosis distribution:")
print(df["diagnosis"].value_counts())


Dataset shape: (569, 32)

First 5 rows:


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678



Column names:
['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']

Data types:
id                           int64
diagnosis                   object
radius_mean                float64
texture_mean               float64
perimeter_mean             float64
area_mean                  float64
smoothness_mean            float64
compactness_mean           float64
concavity_mean             float64
concave points_mean        float64
symmetry_mean              float64
fractal_dimension_mean

In [4]:
# Check duplicate records before cleaning
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["id"].duplicated().sum())

# Remove the identifier column
df = df.drop(columns=["id"])

# Encode diagnosis
df["diagnosis"] = df["diagnosis"].map({
    "B": 0,
    "M": 1
})

# Separate features and target
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts().rename(index={
    0: "Benign",
    1: "Malignant"
}))


Exact duplicate rows: 0
Duplicate IDs: 0

Feature shape: (569, 30)
Target shape: (569,)

Target distribution:
diagnosis
Benign       357
Malignant    212
Name: count, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (455, 30)
X_test: (114, 30)


In [6]:
# Preprocessing for numerical features
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Preprocessing pipeline created successfully.")


Preprocessing pipeline created successfully.


In [7]:
# Logistic Regression pipeline
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

# Random Forest pipeline
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Train both models
logistic_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

print("Both models trained successfully!")


Both models trained successfully!


In [8]:
# Predictions
lr_predictions = logistic_model.predict(X_test)
rf_predictions = rf_model.predict(X_test)

# Logistic Regression evaluation
print("=" * 45)
print("LOGISTIC REGRESSION")
print("=" * 45)

print("Accuracy:",
      round(accuracy_score(y_test, lr_predictions) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    lr_predictions,
    target_names=["Benign", "Malignant"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_predictions))


# Random Forest evaluation
print("\n" + "=" * 45)
print("RANDOM FOREST")
print("=" * 45)

print("Accuracy:",
      round(accuracy_score(y_test, rf_predictions) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    rf_predictions,
    target_names=["Benign", "Malignant"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_predictions))


LOGISTIC REGRESSION
Accuracy: 96.49 %

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.97        72
   Malignant       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114

Confusion Matrix:
[[71  1]
 [ 3 39]]

RANDOM FOREST
Accuracy: 97.37 %

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      1.00      0.98        72
   Malignant       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114

Confusion Matrix:
[[72  0]
 [ 3 39]]


In [9]:
# Create 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Logistic Regression
lr_cv_scores = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

# Random Forest
rf_cv_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("LOGISTIC REGRESSION")
print("Fold accuracies:", lr_cv_scores)
print("Mean CV accuracy:",
      round(lr_cv_scores.mean() * 100, 2), "%")

print("\nRANDOM FOREST")
print("Fold accuracies:", rf_cv_scores)
print("Mean CV accuracy:",
      round(rf_cv_scores.mean() * 100, 2), "%")


LOGISTIC REGRESSION
Fold accuracies: [0.97368421 0.94736842 0.96491228 0.99122807 0.99115044]
Mean CV accuracy: 97.37 %

RANDOM FOREST
Fold accuracies: [0.96491228 0.93859649 0.94736842 0.94736842 0.97345133]
Mean CV accuracy: 95.43 %


In [10]:

import joblib
from pathlib import Path

# Select the best model based on cross-validation
selected_model = logistic_model

# Create the models directory if it doesn't exist
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

# Save the complete pipeline
model_path = model_dir / "breast_cancer_model.pkl"
joblib.dump(selected_model, model_path)

print("Selected model: Logistic Regression")
print("Cross-validation accuracy: 97.37%")
print("Model saved at:", model_path)


Selected model: Logistic Regression
Cross-validation accuracy: 97.37%
Model saved at: ..\models\breast_cancer_model.pkl


In [11]:
loaded_model = joblib.load(model_path)

# Predict one test sample
sample = X_test.iloc[[0]]
actual = y_test.iloc[0]
predicted = loaded_model.predict(sample)[0]

labels = {
    0: "Benign",
    1: "Malignant"
}

print("Actual diagnosis:", labels[int(actual)])
print("Predicted diagnosis:", labels[int(predicted)])


Actual diagnosis: Benign
Predicted diagnosis: Benign


In [12]:
# v1 

In [13]:
import joblib
from pathlib import Path

saved_model = joblib.load(
    Path("../models/breast_cancer_model.pkl")
)

print("Model type:", type(saved_model).__name__)

if hasattr(saved_model, "feature_names_in_"):
    print("Expected features:")
    print(list(saved_model.feature_names_in_))
else:
    print("This model does not expose feature_names_in_.")

Model type: Pipeline
Expected features:
['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']


In [14]:
# v2 

In [15]:
import joblib
import numpy as np
from pathlib import Path
from collections import Counter
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

saved_model = joblib.load(
    Path("../models/breast_cancer_model.pkl")
)

expected_features = list(saved_model.feature_names_in_)

print("Test data shape:", X_test.shape)
print("Expected feature count:", len(expected_features))
print(
    "Feature sets match:",
    set(X_test.columns) == set(expected_features)
)

if set(X_test.columns) != set(expected_features):
    raise ValueError(
        "X_test columns do not match the saved model features."
    )

# Align the feature order with the saved model
X_test_aligned = X_test[expected_features]

# Predict using the saved model
predictions = saved_model.predict(X_test_aligned)

print("\nPrediction counts:")
print(Counter(predictions))

print("\nActual test-label counts:")
print(Counter(np.asarray(y_test).ravel()))

print("\nSaved model accuracy:")
print(f"{accuracy_score(y_test, predictions) * 100:.2f}%")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, predictions))

print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))

Test data shape: (114, 30)
Expected feature count: 30
Feature sets match: True

Prediction counts:
Counter({np.int64(0): 74, np.int64(1): 40})

Actual test-label counts:
Counter({np.int64(0): 72, np.int64(1): 42})

Saved model accuracy:
96.49%

Confusion matrix:
[[71  1]
 [ 3 39]]

Classification report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.97        72
           1       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [16]:
# v3 

In [19]:
import pandas as pd

expected_features = list(saved_model.feature_names_in_)
X_test_aligned = X_test[expected_features]

predictions = saved_model.predict(X_test_aligned)

results = pd.DataFrame({
    "actual": np.asarray(y_test).ravel(),
    "predicted": predictions
}, index=X_test_aligned.index)

for label in [0, 1]:
    correct_samples = results[
        (results["actual"] == label) &
        (results["predicted"] == label)
    ]

    if correct_samples.empty:
        print(f"No correctly predicted sample for class {label}")
        continue

    sample_index = correct_samples.index[0]

    print(f"\n{'=' * 12} SAMPLE: CLASS {label} {'=' * 12}")
    print(X_test_aligned.loc[sample_index].to_dict())
    print("Actual label:", results.loc[sample_index, "actual"])
    print("Saved model prediction:", results.loc[sample_index, "predicted"])


============ SAMPLE: CLASS 0 ============
{'radius_mean': 11.41, 'texture_mean': 10.82, 'perimeter_mean': 73.34, 'area_mean': 403.3, 'smoothness_mean': 0.09373, 'compactness_mean': 0.06685, 'concavity_mean': 0.03512, 'concave points_mean': 0.02623, 'symmetry_mean': 0.1667, 'fractal_dimension_mean': 0.06113, 'radius_se': 0.1408, 'texture_se': 0.4607, 'perimeter_se': 1.103, 'area_se': 10.5, 'smoothness_se': 0.00604, 'compactness_se': 0.01529, 'concavity_se': 0.01514, 'concave points_se': 0.00646, 'symmetry_se': 0.01344, 'fractal_dimension_se': 0.002206, 'radius_worst': 12.82, 'texture_worst': 15.97, 'perimeter_worst': 83.74, 'area_worst': 510.5, 'smoothness_worst': 0.1548, 'compactness_worst': 0.239, 'concavity_worst': 0.2102, 'concave points_worst': 0.08958, 'symmetry_worst': 0.3016, 'fractal_dimension_worst': 0.08523}
Actual label: 0
Saved model prediction: 0

============ SAMPLE: CLASS 1 ============
{'radius_mean': 20.94, 'texture_mean': 23.56, 'perimeter_mean': 138.9, 'area_mean': 